# Description

In this notebook, I will explore the different between different hierarchical quantization, including:
- Vector quantization.
- Matrix quantization.
- Tensor quantization.

In [1]:
import os 
import torch 
import bitsandbytes as bnb

In [2]:
def quantization_error_l2_norm(original, dequantized):
    """
    Compute the relative error between the original and dequantized tensors using l2 norm.
    """
    return torch.norm(original - dequantized)


def quantization_error_mse(original, dequantized):
    """
    Compute the Mean Squared Error (MSE) between the original and dequantized tensors.
    """
    return torch.mean((original - dequantized) ** 2)


def quantization_error_kl_divergence(original, dequantized, num_bins=2048, epsilon=1e-10):
    """
    Compute the KL divergence between the distributions of the original and dequantized tensors with float16.
    """
    orig_hist = torch.histc(original.float(), bins=num_bins, min=-1.0, max=1.0)
    deq_hist = torch.histc(dequantized.float(), bins=num_bins, min=-1.0, max=1.0)

    orig_prob = orig_hist / (torch.sum(orig_hist) + epsilon)
    deq_prob = deq_hist / (torch.sum(deq_hist) + epsilon)

    kl_div = torch.sum(orig_prob * torch.log((orig_prob + epsilon) / (deq_prob + epsilon)))
    return kl_div

# 1. Vector quantization

In [3]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.to(torch.float32)
    return q_mat, scales

def de_quantize_row_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[:, None]
    output = output.to(out_dtype)
    return output

In [4]:
d_type = torch.float16
N = 512

W = torch.randn(N, N, device='cuda', dtype=d_type)

In [5]:
W_q, w_scale = quantize_row_matrix_int8_symmetric(W)

print(f"W_q shape: {W_q.shape}, dtype: {W_q.dtype}")
print(f"w_scale shape: {w_scale.shape}, dtype: {w_scale.dtype}")

W_q shape: torch.Size([512, 512]), dtype: torch.int8
w_scale shape: torch.Size([512]), dtype: torch.float32


In [6]:
W_deq = de_quantize_row_matrix_int8_symmetric(W_q, w_scale)
print(f"Shape of A_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of A_deq: torch.Size([512, 512]), dtype: torch.float16


In [7]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !!")
else:
    print("WRONG - Dequantized matrix is NOT close !!")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (row-wise symmetric int8): {error_l2.item():.6f}")
    
error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (row-wise symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (row-wise symmetric int8): {error_kl.item():.6f}")

Correct !!
Quantization L2 norm error (row-wise symmetric int8): 3.791016
Quantization MSE (row-wise symmetric int8): 0.000055
Quantization KL divergence (row-wise symmetric int8): 0.482239


# 2. Matrix quantization

In [8]:
def quantize_matrix_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    scale = torch.tensor(scale, dtype=torch.float32)
    return q_mat, scale

def de_quantize_matrix_symmetric_int8(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 tensor to float.
    q_mat: input int8 tensor
    scale: scale factor (single value)
    """
    output = q_mat.to(torch.float32) 
    output = output * scale
    output = output.to(out_dtype)
    return output

In [9]:
d_type = torch.float16
N = 1024

W = torch.randn(N, N, device='cuda', dtype=d_type)
print(f"W shape: {W.shape}, dtype: {W.dtype}")

W shape: torch.Size([1024, 1024]), dtype: torch.float16


In [10]:
q_W, scales = quantize_matrix_symmetric_int8(W)
print(f"Shape of q_W: {q_W.shape}, Shape of scales: {scales.shape}")

Shape of q_W: torch.Size([1024, 1024]), Shape of scales: torch.Size([])


/tmp/ipykernel_579374/2675896001.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [11]:
W_deq = de_quantize_matrix_symmetric_int8(q_W, scales)
print(f"Shape of W_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of W_deq: torch.Size([1024, 1024]), dtype: torch.float16


In [12]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (symmetric int8): 11.414062
Quantization MSE (symmetric int8): 0.000124
Quantization KL divergence (symmetric int8): 14.934549


# 3. Tensor quantization

In [13]:
def quantize_tensor_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    scale = torch.tensor(scale, dtype=torch.float32)
    return q_mat, scale

def de_quantize_tensor_symmetric_int8(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 tensor to float.
    q_mat: input int8 tensor
    scale: scale factor (single value)
    """
    output = q_mat.to(torch.float32) 
    output = output * scale
    output = output.to(out_dtype)
    return output

In [14]:
H = 16
N = 512
d_type = torch.float16

W = torch.randn((H, N, N), device='cuda', dtype=d_type)
print(f"W shape: {W.shape}, dtype: {W.dtype}")

W shape: torch.Size([16, 512, 512]), dtype: torch.float16


In [15]:
W_q, scales = quantize_tensor_symmetric_int8(W)
print(f"Shape of W_q: {W_q.shape}, Shape of scales: {scales.shape}")

Shape of W_q: torch.Size([16, 512, 512]), Shape of scales: torch.Size([])


/tmp/ipykernel_579374/315809518.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [16]:
W_deq = de_quantize_tensor_symmetric_int8(W_q, scales)
print(f"Shape of W_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of W_deq: torch.Size([16, 512, 512]), dtype: torch.float16


In [17]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (symmetric int8): 24.203125
Quantization MSE (symmetric int8): 0.000140
Quantization KL divergence (symmetric int8): 14.956940


# 4. Matmul - Vector quantization

In [18]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.to(torch.float32)
    return q_mat, scales

def de_quantize_row_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[:, None]
    output = output.to(out_dtype)
    return output

def quantized_column_matrix_int_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-column basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=0, keepdim=True)  # shape (1, M)
    scales = (max_vals / qmax).squeeze(0)  # shape (M,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(0)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales


def de_quantized_column_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-column basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each column (shape (M,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[None, :]
    output = output.to(out_dtype)
    return output

def de_quantize_row_matrix_int8_symmetric_matmul(q_mat:torch.Tensor, x_scale:torch.Tensor, w_scale: torch.Tensor,\
                                        out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * x_scale[:, None] * w_scale[None, :]
    output = output.to(out_dtype)
    return output

def dummy_int8_matmul(A_int8:torch.Tensor, B_int:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    result_float = torch.matmul(A_int8.float(), B_int.float())
    return result_float.to(out_dtype)

In [19]:
N = 1024
M = 1024
P = 512

X = torch.randn(M, N, device='cuda', dtype=d_type)
W = torch.randn(N, P, device='cuda', dtype=d_type)

A = torch.matmul(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")

Shape of A: torch.Size([1024, 512]), dtype: torch.float16


In [20]:
X_q, x_scale = quantize_row_matrix_int8_symmetric(X)
print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")

W_q, w_scale = quantized_column_matrix_int_symmetric(W)
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of x_scale: torch.Size([1024]), dtype: torch.float32
Shape of W_q: torch.Size([1024, 512]), dtype: torch.int8
Shape of w_scale: torch.Size([512]), dtype: torch.float32


In [21]:
# A_q = bnb.functional.int8_linear_matmul(X_q, W_q)
A_q = torch.matmul(X_q.float(), W_q.float())  # TODO
print(f"Shape of A_q: {A_q.shape}, dtype: {A_q.dtype}")

Shape of A_q: torch.Size([1024, 512]), dtype: torch.float32


In [22]:
A_deq = de_quantize_row_matrix_int8_symmetric_matmul(A_q, x_scale, w_scale)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([1024, 512]), dtype: torch.float16


In [23]:
if torch.allclose(A, A_deq, rtol=1.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (hierarchical quantization int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (hierarchical quantization int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (hierarchical quantization int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (hierarchical quantization int8): 257.500000
Quantization MSE (hierarchical quantization int8): 0.126343
Quantization KL divergence (hierarchical quantization int8): 0.218174


# 5. Matmul - Matrix quantization

In [24]:
N = 1024
d_type = torch.float16

W = torch.randn(N, N, device='cuda', dtype=d_type)
X = torch.randn(N, N, device='cuda', dtype=d_type)

A = torch.matmul(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")
print(f"Min and Max of A: {torch.min(A).item():.6f}, {torch.max(A).item():.6f}")

Shape of A: torch.Size([1024, 1024]), dtype: torch.float16
Min and Max of A: -167.375000, 155.250000


In [25]:
X_q, x_scale = quantize_matrix_symmetric_int8(X)
W_q, w_scale = quantize_matrix_symmetric_int8(W)

print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of x_scale: torch.Size([]), dtype: torch.float32
Shape of W_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of w_scale: torch.Size([]), dtype: torch.float32


/tmp/ipykernel_579374/2675896001.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [26]:
A_deq = torch.matmul(X_q.float(), W_q.float())  * x_scale * w_scale  # TODO 
A_deq = A_deq.to(d_type)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([1024, 1024]), dtype: torch.float16


In [27]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (matrix symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (matrix symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (matrix symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (matrix symmetric int8): 500.250000
Quantization MSE (matrix symmetric int8): 0.238647
Quantization KL divergence (matrix symmetric int8): 4.498326


# 6. Matmul - Tensor quantization

In [28]:
H = 32
N = 512

W = torch.randn((H, N, N), device='cuda', dtype=d_type)
X = torch.randn((H, N, N), device='cuda', dtype=d_type)

A = torch.bmm(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")

Shape of A: torch.Size([32, 512, 512]), dtype: torch.float16


In [29]:
X_q, x_scale = quantize_tensor_symmetric_int8(X)
W_q, w_scale = quantize_tensor_symmetric_int8(W)

print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([32, 512, 512]), dtype: torch.int8
Shape of x_scale: torch.Size([]), dtype: torch.float32
Shape of W_q: torch.Size([32, 512, 512]), dtype: torch.int8
Shape of w_scale: torch.Size([]), dtype: torch.float32


/tmp/ipykernel_579374/315809518.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [30]:
A_deq = torch.bmm(X_q.float(), W_q.float())  * x_scale * w_scale
A_deq = A_deq.to(d_type)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([32, 512, 512]), dtype: torch.float16


In [31]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (tensor symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (tensor symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (tensor symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (tensor symmetric int8): 1139.000000
Quantization MSE (tensor symmetric int8): 0.154785
Quantization KL divergence (tensor symmetric int8): 6.730340


# 7. Test Query - Key in Attention

In [32]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    qmin, qmax = -128, 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales


def quantized_column_matrix_int_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-column basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    qmin, qmax = -128, 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=0, keepdim=True)  # shape (1, M)
    scales = (max_vals / qmax).squeeze(0)  # shape (M,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(0)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales


def de_quantize_row_matrix_int8_symmetric_matmul(q_mat:torch.Tensor, x_scale:torch.Tensor, w_scale: torch.Tensor,\
                                        out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * x_scale[:, None] * w_scale[None, :]
    output = output.to(out_dtype)
    return output

def dummy_int8_matmul(A_int8:torch.Tensor, B_int:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    result_float = torch.matmul(A_int8.float(), B_int.float())
    return result_float.to(out_dtype)

def dummy_int32_matmul(A_int32:torch.Tensor, B_int32:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int32.dtype != torch.int32 or B_int32.dtype != torch.int32:
        raise ValueError("Both A and B must be int32 tensors.")
    result_float = torch.matmul(A_int32.float(), B_int32.float())
    return result_float.to(out_dtype)

In [102]:
L = 50
D_MODEL = 64
d_type = torch.float16

X = torch.randn(L, D_MODEL, device='cuda', dtype=d_type)
W_q = torch.randn(D_MODEL, D_MODEL, device='cuda', dtype=d_type)
W_k = torch.randn(D_MODEL, D_MODEL, device='cuda', dtype=d_type)

Q = torch.matmul(X, W_q)
K = torch.matmul(X, W_k)
score = torch.matmul(Q, K.transpose(-2, -1))  # TODO remember to transpose K 
print(f"Shape of score: {score.shape}, dtype: {score.dtype}")

Shape of score: torch.Size([50, 50]), dtype: torch.float16


In [103]:
Q

tensor([[-2.5215e+00, -4.6289e+00, -6.0508e+00,  ..., -1.4014e+00,
         -1.3096e+00, -9.4609e+00],
        [ 1.6260e+00,  7.2109e+00,  1.2891e+00,  ...,  8.6133e-01,
          3.6113e+00, -8.6250e+00],
        [ 8.0625e+00,  6.8398e+00, -2.7078e+01,  ...,  1.2508e+01,
          5.5391e+00,  3.8047e+00],
        ...,
        [-4.5195e+00,  1.2344e+01,  3.6152e+00,  ...,  6.2578e+00,
          1.1719e-02, -1.3926e+00],
        [-1.3652e+00, -1.1242e+01,  4.5685e-02,  ...,  1.7219e+01,
          2.3047e+00,  1.1047e+01],
        [-6.3906e+00, -7.4062e+00,  1.0031e+01,  ...,  3.6694e-01,
          2.5742e+00,  1.2471e+00]], device='cuda:0', dtype=torch.float16)

In [104]:
K

tensor([[  3.0840,  -6.8438,  -6.6992,  ...,   9.7656,   3.7324,  -1.4561],
        [ 11.0859,   8.6250,   8.1250,  ..., -15.2656,   2.3066,  -3.9688],
        [  5.7109,   0.1866,  -6.7969,  ..., -10.3203,  -8.4531,  -0.0604],
        ...,
        [  4.2500,  -6.3125,   6.4648,  ...,   1.1133,   1.5801,   7.3359],
        [  1.6709, -17.5312,  -7.7578,  ..., -11.7656, -16.1719,  12.8984],
        [  2.8633,  -3.6973,   9.7656,  ...,   3.2715,   9.7188, -10.0312]],
       device='cuda:0', dtype=torch.float16)

In [105]:
score 

tensor([[  568.0000,  -241.2500,  -504.7500,  ...,  -116.0000,  -591.5000,
           317.0000],
        [ -212.3750,   535.0000,    49.5625,  ...,    57.2500,  -190.6250,
          -184.6250],
        [  173.5000,   860.0000,  -505.5000,  ...,  -629.0000,   616.5000,
           -57.5000],
        ...,
        [ -103.3750,   165.0000,  -489.7500,  ...,   -97.1875,  -361.2500,
          -110.6875],
        [  397.7500,    95.4375, -1236.0000,  ...,   576.5000,   783.5000,
          -431.5000],
        [ -196.6250,  -753.0000,  -923.0000,  ...,   -40.3438,   106.3750,
           966.0000]], device='cuda:0', dtype=torch.float16)

In [106]:
X_q, x_scale = quantize_row_matrix_int8_symmetric(X)
print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")

W_q_q, w_q_scale = quantized_column_matrix_int_symmetric(W_q)
print(f"Shape of W_q: {W_q_q.shape}, dtype: {W_q_q.dtype}")
print(f"Shape of w_q_scale: {w_q_scale.shape}, dtype: {w_q_scale.dtype}")

W_k_q, w_k_scale = quantized_column_matrix_int_symmetric(W_k)
print(f"Shape of W_k: {W_k_q.shape}, dtype: {W_k_q.dtype}")
print(f"Shape of w_k_scale: {w_k_scale.shape}, dtype: {w_k_scale.dtype}")

Shape of X_q: torch.Size([50, 64]), dtype: torch.int8
Shape of x_scale: torch.Size([50]), dtype: torch.float32
Shape of W_q: torch.Size([64, 64]), dtype: torch.int8
Shape of w_q_scale: torch.Size([64]), dtype: torch.float32
Shape of W_k: torch.Size([64, 64]), dtype: torch.int8
Shape of w_k_scale: torch.Size([64]), dtype: torch.float32


In [107]:
# Q_q = dummy_int8_matmul(X_q, W_q_q)
# K_q = dummy_int8_matmul(X_q, W_k_q)

# Q_deq = Q_q.to(float) * x_scale[:, None] * w_q_scale[None, :]
# Q_deq = Q_deq.to(d_type)
# K_deq = K_q.to(float) * x_scale[:, None] * w_k_scale[None, :]
# K_deq = K_deq.to(d_type)

# score_deq = torch.matmul(Q_deq, K_deq.transpose(-2, -1))
# print(f"Shape of score_deq: {score_deq.shape}, dtype: {score_deq.dtype} \n")

# if torch.allclose(score, score_deq, rtol=2.0, atol=2.0):
#     print("Correct !! \n")
# else:
#     print("WRONG - Dequantized matrix is NOT close !! \n")

In [110]:
Q_q = dummy_int8_matmul(X_q, W_q_q)
K_q = dummy_int8_matmul(X_q, W_k_q)

global_w_q_scale = torch.mean(w_q_scale)
global_w_k_scale = torch.mean(w_k_scale)
global_x_scale = torch.mean(x_scale)

Q_deq = Q_q.to(float) * x_scale[:, None] * global_w_q_scale
# Q_deq = Q_q.to(float) * x_scale[:, None] * w_q_scale[None, :]
# Q_deq = Q_q.to(float) * global_x_scale * w_q_scale[None, :]
Q_deq = Q_deq.to(d_type)

K_deq = K_q.to(float) * x_scale[:, None] * global_w_k_scale
# K_deq = K_q.to(float) * x_scale[:, None] * w_k_scale[None, :]
# K_deq = K_q.to(float) * global_x_scale * w_k_scale[None, :]
K_deq = K_deq.to(d_type)

score_deq = torch.matmul(Q_deq, K_deq.transpose(-2, -1))
print(f"Shape of score_deq: {score_deq.shape}, dtype: {score_deq.dtype} \n")

if torch.allclose(score, score_deq, rtol=5.0, atol=5.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")

Shape of score_deq: torch.Size([50, 50]), dtype: torch.float16 

WRONG - Dequantized matrix is NOT close !! 

